<a href="https://colab.research.google.com/github/shubhs77712/shubhs77712/blob/main/shubhamthakur.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os

folders = ["data", "model", "chatbot", "utils"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    open(f"{folder}/__init__.py", "w").close()


In [4]:
%%writefile data/data_loader.py
import yfinance as yf

def get_stock_data(ticker, period="1y"):
    stock = yf.Ticker(ticker)
    df = stock.history(period=period)

    if df.empty:
        raise ValueError("Invalid ticker symbol")

    return df

Writing data/data_loader.py


In [5]:
import os

folders = ["data", "model", "utils"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    open(f"{folder}/__init__.py", "w").close()

In [6]:
%%writefile data/data_loader.py
import yfinance as yf

def get_stock_data(ticker, period="1y"):
    stock = yf.Ticker(ticker)
    df = stock.history(period=period)

    if df.empty:
        raise ValueError("Invalid ticker")

    return df

Overwriting data/data_loader.py


In [7]:
%%writefile utils/preprocessing.py
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def preprocess_data(data, time_steps=60):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data.reshape(-1, 1))

    X, y = [], []

    for i in range(time_steps, len(scaled_data)):
        X.append(scaled_data[i-time_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    return X, y, scaler

Writing utils/preprocessing.py


In [8]:
%%writefile utils/preprocessing.py
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def preprocess_data(data, time_steps=60):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data.reshape(-1, 1))

    X, y = [], []

    for i in range(time_steps, len(scaled_data)):
        X.append(scaled_data[i-time_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    return X, y, scaler

Overwriting utils/preprocessing.py


In [9]:
%%writefile model/train.py
from data.data_loader import get_stock_data
from utils.preprocessing import preprocess_data
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

def train_model(ticker):
    df = get_stock_data(ticker)
    close_prices = df["Close"].values

    X, y, scaler = preprocess_data(close_prices)

    model = Sequential()
    model.add(LSTM(50, input_shape=(X.shape[1], 1)))
    model.add(Dense(1))

    model.compile(optimizer="adam", loss="mse")
    model.fit(X, y, epochs=3, verbose=0)

    model.save(f"model/{ticker}_model.h5")

    return model, scaler

Writing model/train.py


In [10]:
%%writefile data/data_loader.py
import yfinance as yf

def get_stock_data(ticker, period="1y"):
    stock = yf.Ticker(ticker)
    df = stock.history(period=period)

    if df.empty:
        raise ValueError("Invalid ticker")

    return df

Overwriting data/data_loader.py


In [11]:
%%writefile utils/preprocessing.py
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def preprocess_data(data, time_steps=60):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data.reshape(-1, 1))

    X, y = [], []

    for i in range(time_steps, len(scaled_data)):
        X.append(scaled_data[i-time_steps:i, 0])
        y.append(scaled_data[i, 0])

    X, y = np.array(X), np.array(y)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    return X, y, scaler

Overwriting utils/preprocessing.py


In [12]:
%%writefile app.py
import streamlit as st
from data.data_loader import get_stock_data
from model.predict import predict_stock
from chatbot.intent import detect_intent
from chatbot.response import generate_response
import plotly.graph_objects as go

st.title("AI Stock Advisor")

user_input = st.text_input("Ask something:")

if user_input:
    intent, ticker = detect_intent(user_input)

    if intent == "trend":
        data = get_stock_data(ticker)
        st.line_chart(data["Close"])

    elif intent == "predict":
        pred, conf = predict_stock(ticker)
        st.write(pred, conf)

Writing app.py


In [13]:
%%writefile model/predict.py
import numpy as np
from tensorflow.keras.models import load_model
from data.data_loader import get_stock_data
from utils.preprocessing import preprocess_data

def predict_stock(ticker, days=7):
    df = get_stock_data(ticker)
    close_prices = df["Close"].values

    X, y, scaler = preprocess_data(close_prices)

    try:
        model = load_model(f"model/{ticker}_model.h5")
    except:
        from model.train import train_model
        model, scaler = train_model(ticker)

    last_sequence = X[-1]
    predictions = []

    current_input = last_sequence

    for _ in range(days):
        pred = model.predict(current_input.reshape(1, -1, 1), verbose=0)[0][0]
        predictions.append(pred)
        current_input = np.append(current_input[1:], pred)

    predictions = scaler.inverse_transform(np.array(predictions).reshape(-1, 1))

    confidence = 0.8
    return predictions.flatten(), confidence

Writing model/predict.py


In [14]:
%%writefile chatbot/intent.py
def detect_intent(text):
    text = text.upper()
    words = text.split()

    ticker = None
    for word in words:
        if len(word) <= 5 and word.isalpha():
            ticker = word

    if "PREDICT" in text:
        return "predict", ticker
    elif "TREND" in text:
        return "trend", ticker
    return "unknown", ticker

Writing chatbot/intent.py


In [15]:
%%writefile chatbot/response.py
def generate_response(intent, ticker, predictions=None, confidence=None):
    if intent == "trend":
        return f"Showing trend for {ticker}"
    elif intent == "predict":
        return f"Predictions: {predictions}, Confidence: {confidence}"
    return "Unknown request"

Writing chatbot/response.py


In [ ]:
from pyngrok import ngrok
!streamlit run app.py &

public_url = ngrok.connect(port='8501')
public_url

In [ ]:
from pyngrok import ngrok
!pkill streamlit
!streamlit run app.py &

url = ngrok.connect(8501)
print(url)




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.16.233:8501



In [17]:
!pip install tensorflow


In [18]:
!pip install streamlit yfinance pandas numpy plotly scikit-learn tensorflow pyngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 68.1 MB/s eta 0:00:00


In [19]:
!pip install yfinance scikit-learn tensorflow

In [20]:
!pip install pyngrok
